# LiViFuser validation-only uncertainty score freeze (T4 x2)

Attach exactly four Kaggle datasets: (1) the validation-code bundle, (2) all 18 files from gpu_handoff_train_val_v1, (3) livifuser_dinov3_splus_cache_v2_bundle.zip, and (4) livifuser_simulation_sweep_v1_results.zip. Do not attach either held-out handoff or held-out cache. Select GPU T4 x2 and run all cells. This notebook performs no training and fails closed if it sees a held-out filename or identity.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

INPUT = Path('/kaggle/input')
WORK = Path('/kaggle/working')
AMENDMENT_SHA = '8760474F1CCC6269BD23A28489DD01076891ECBF9E66A6F39BBF8E2838F6DCD7'
def find_validation_manifests(root):
    return [
        path for path in root.rglob('cloud_bundle_manifest.json')
        if json.loads(path.read_text()).get('frozen_amendment_sha256') == AMENDMENT_SHA
    ]
manifests = find_validation_manifests(INPUT)
if not manifests:
    archives = list(INPUT.rglob('livifuser_sim_validation_code_*.zip'))
    assert len(archives) == 1, f'expected one validation-code archive, found {len(archives)}'
    code_extract = WORK / 'livifuser_validation_code'
    assert not code_extract.exists(), f'refusing existing extraction root: {code_extract}'
    with zipfile.ZipFile(archives[0]) as archive:
        for info in archive.infolist():
            member = PurePosixPath(info.filename)
            assert not member.is_absolute() and '..' not in member.parts, f'unsafe code member: {info.filename}'
        archive.extractall(code_extract)
    manifests = find_validation_manifests(code_extract)
assert len(manifests) == 1, f'expected one code manifest, found {len(manifests)}'
REPO = manifests[0].parent
sys.path.insert(0, str(REPO / 'src'))
sys.path.insert(0, str(REPO / 'scripts'))
from prepare_sim_training_data import refuse_heldout
from replay_sim_validation_scores import validate_result_source

from livifuser_nav.cloud_bundle import verify_cloud_bundle

verification = verify_cloud_bundle(REPO)
config = REPO / 'config/simulation_sweep_v1.json'
amendment = REPO / 'docs/experiments/PREREGISTRATION_SIM_EVALUATION_EXECUTION_AMENDMENT_2026-08-24.md'
audit_report = REPO / 'artifacts/simulation_sweep_v1_result_audit.json'
expected = {
    config: '76680BCE45A67D5D91F42660D5EC25F90450B281838CE311E329477C4E36F09E',
    amendment: '8760474F1CCC6269BD23A28489DD01076891ECBF9E66A6F39BBF8E2838F6DCD7',
    audit_report: '4D1CEA8F2D61EF76E1A48770FB6228F14683DAF6943C4932C06FCE0FB46611B3',
}
for path, expected_sha in expected.items():
    assert hashlib.sha256(path.read_bytes()).hexdigest().upper() == expected_sha, f'hash drift: {path}'
compile_env = os.environ.copy()
compile_env['PYTHONPYCACHEPREFIX'] = str(WORK / 'livifuser_validation_pycache')
subprocess.run([sys.executable, '-m', 'compileall', '-q', str(REPO / 'src'), str(REPO / 'scripts')], check=True, env=compile_env)
print(json.dumps({'repository': str(REPO), 'cloud_bundle': verification}, indent=2))

In [ ]:
import torch

assert torch.cuda.is_available(), 'enable the Kaggle GPU T4 x2 accelerator'
devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert len(devices) == 2 and all('T4' in name for name in devices), f'expected T4 x2, found {devices}'
print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, 'devices': devices}, indent=2))

In [ ]:
# This filename-level guard runs before result/cache discovery or hashing.
refuse_heldout(INPUT)
result_sources = [
    path for path in INPUT.rglob('livifuser_simulation_sweep_v1')
    if path.is_dir() and (path / 'summary.json').is_file()
]
if not result_sources:
    result_sources = list(INPUT.rglob('livifuser_simulation_sweep_v1_results.zip'))
assert len(result_sources) == 1, f'expected one expanded result root, found {len(result_sources)}'
RESULT_SOURCE = result_sources[0]
result_verification = validate_result_source(
    RESULT_SOURCE, json.loads(audit_report.read_text())
)
DATA_ROOT = WORK / 'livifuser_sim_validation_data_v1'
DATA_PLAN = WORK / 'livifuser_sim_validation_data_plan_v1.json'
prepare_command = [
    sys.executable, str(REPO / 'scripts/prepare_sim_training_data.py'),
    '--input-root', str(INPUT), '--work-root', str(DATA_ROOT),
    '--plan-output', str(DATA_PLAN), '--validation-only',
]
print(' '.join(prepare_command))
subprocess.run(prepare_command, cwd=REPO, check=True)
plan = json.loads(DATA_PLAN.read_text())
assert 'train' not in plan and plan['heldout_attached'] is False
assert plan['validation']['episode_count'] == 30
assert plan['validation']['accepted_samples'] == 13125
assert plan['validation']['windows_k8_h8'] == 9459
print(json.dumps({'result_source': result_verification, 'validation': plan['validation']['episode_count']}, indent=2))

In [ ]:
OUTPUT_ROOT = WORK / 'livifuser_sim_validation_score_freeze_v1'
BUNDLE = WORK / 'livifuser_sim_validation_score_freeze_v1_bundle.zip'
run_command = [
    sys.executable, str(REPO / 'scripts/run_sim_validation_score_freeze_kaggle.py'),
    '--data-plan', str(DATA_PLAN), '--results-archive', str(RESULT_SOURCE),
    '--audit-report', str(audit_report), '--config', str(config),
    '--output-root', str(OUTPUT_ROOT), '--bundle-output', str(BUNDLE),
    '--cuda-device', '0', '--cuda-device', '1',
]
print(' '.join(run_command))
subprocess.run(run_command, cwd=REPO, check=True)
with zipfile.ZipFile(BUNDLE) as archive:
    manifest = json.loads(archive.read('SCORE_FREEZE_MANIFEST.json'))
    completion = json.loads(archive.read('SCORE_FREEZE_COMPLETE.json'))
assert manifest['status'] == 'FROZEN_VALIDATION_ONLY'
assert completion['status'] == 'COMPLETE' and completion['heteroscedastic_record_count'] == 21
bundle_sha = hashlib.sha256(BUNDLE.read_bytes()).hexdigest().upper()
print(json.dumps({'download': str(BUNDLE), 'size_bytes': BUNDLE.stat().st_size, 'sha256': bundle_sha, 'records': 21, 'closed_loop_thresholds': 12}, indent=2))